---
exec_all: true
---

# Interactive map visualization of CML vector data

> This module provides interactive map visualization for Commercial Microwave Link (CML) networks using Folium and GeoPandas. It allows you to explore link geometries on a map with various basemaps and controls.

:::{.callout-warning}
Scroll wheel zoom is disabled in the documentation examples below to make page navigation easier. When you create maps yourself, scroll wheel zoom will be enabled by default.
:::

In [ ]:
#| default_exp maps._vector

In [ ]:
#| exec_doc
#| export
from typing import Callable, Literal

import xarray as xr
import pandas as pd
import geopandas as gpd
import folium
import folium.plugins
from shapely.geometry import LineString, Point
from matplotlib.colors import Normalize, to_hex, ListedColormap
import matplotlib.pyplot as plt

from raincell.data.convert import convert_sublinks_to_gdf
from raincell.maps._helpers import get_center, get_zoom_start

In [ ]:
#| exec_doc
#| hide
from raincell import open_cml_sample, open_gauge_sample

In [ ]:
#| exec_doc
cml, gauges = open_cml_sample(), open_gauge_sample()

## Default map

Create a Folium map with multiple basemap options (OpenStreetMap, OpenTopoMap, Esri World Imagery), layer controls, fullscreen toggle, geocoder search, and measurement tools that will be used as a default for exploration.

In [ ]:
#| exec_doc
#| export
def setup_default_map(
        m: folium.Map = None, # Optional folium map to be setup with basemaps and controls
        show: Literal["OpenStreetMap", "OpenTopoMap", "Esri.WorldImagery", "Esri.WorldGrayCanvas"] = "Esri.WorldGrayCanvas" # Basemap to show by default.
        ) -> folium.Map:
    """ Create a default folium map with common basemaps, layers and controls."""
    m = m or folium.Map(tiles=None)
    for tile in ["OpenStreetMap", "OpenTopoMap", "Esri.WorldImagery", "Esri.WorldGrayCanvas"]:
        folium.TileLayer(tile, name=tile, show=(tile==show)).add_to(m)
    folium.LayerControl().add_to(m)
    folium.plugins.Fullscreen(position="topright",force_separate_button=True).add_to(m)
    folium.plugins.Geocoder(collapsed=True, add_marker=False).add_to(m)
    folium.plugins.MeasureControl(position="bottomright").add_to(m)
    return m

In [ ]:
#| exec_doc
m = setup_default_map()

In [ ]:
#| exec_doc
#| echo: false
m.options['scrollWheelZoom'] = False
m

## Visualize Gauges on an interactive map.

In [ ]:
#| exec_doc
#| export
def explore_gauges(
    gauges: xr.Dataset | xr.DataArray, # Gauges in pws OpenSense standard
    default: Callable = setup_default_map, # Function to set up a default map including for example basemaps or controls
    **explore_kwargs
    ) -> folium.Map:
    gauges_df = gauges.id.to_dataframe()
    gauges_gdf = gpd.GeoDataFrame(gauges_df[[c for c in gauges_df.columns if c not in ["lat", "lon", "id"]]], geometry=[Point(g.lon, g.lat) for _, g in gauges_df.iterrows()], crs="EPSG:4326")
    explore_kwargs = {"marker_type": "circle", "marker_kwds": {"radius": 75, "fill": True, "color": "black", "fillColor": "yellow"}, **explore_kwargs}
    m = gauges_gdf.explore(tiles=None, **explore_kwargs)
    gauges_layer = [c for c in m._children.values() if isinstance(c, folium.features.GeoJson)][-1]
    gauges_layer.layer_name = "Gauges"
    if default:
        m = default(m)
    folium.plugins.Search(layer=gauges_layer, geom_type="Point", placeholder="Gauge ID", collapsed=True, search_label="id").add_to(m)
    return m

In [ ]:
#| exec_doc
m = explore_gauges(gauges)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

## Visualize CML links on an interactive map. 

Each link is drawn as a line between its two antenna sites. Includes a search control to find links by ID. It also includes right-click to copy `cml_id` to clipboard. Then you only need to paste it wherever you require.

In [ ]:
#| exporti
def rclick_cp_cml_id(m: folium.Map):
    """ Copy cml_id to clipboard when right clicking on a link """
    js_code = """
    <script>
    document.addEventListener('DOMContentLoaded', function() {
        setTimeout(function() {
            var map = Object.values(window).find(v => v instanceof L.Map);
            map.eachLayer(function(layer) {
                if (layer.feature && layer.feature.properties && layer.feature.properties.cml_id) {
                    layer.on('contextmenu', function(e) {
                        L.DomEvent.stopPropagation(e);
                        L.DomEvent.preventDefault(e);
                        var cmlId = e.target.feature.properties.cml_id;
                        // var jsonObj = JSON.stringify(cmlId);
                        navigator.clipboard.writeText(cmlId);
                        // console.log('Copied:', jsonObj);
                        console.log('Copied:', cmlId);
                    });
                }
            });
        }, 1000);
    });
    </script>
    """

    m.get_root().html.add_child(folium.Element(js_code))

In [ ]:
#| exec_doc
#| export
def explore_links(
    cml: xr.Dataset | xr.DataArray, # CML in OpenSense standard
    default: Callable = setup_default_map, # Function to set up a default map including for example basemaps or controls
    cp_cml_id_on_rclick: bool = True, # Enable copying cml_id to clipboard when right clicking on a link
    **explore_kwargs
    ) -> folium.Map:
    """ Geopandas [explore](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html) wrapper to visualize CML link structure """
    links_df = cml.cml_id.to_dataframe().drop_duplicates()
    geom = [LineString([(l.site_0_lon, l.site_0_lat), (l.site_1_lon, l.site_1_lat)]) for _, l in links_df.iterrows()]
    links_gdf = gpd.GeoDataFrame(links_df["length"], geometry=geom, crs="EPSG:4326")
    m = links_gdf.explore(tiles=None, **explore_kwargs)
    link_layer, = [c for c in m._children.values() if isinstance(c, folium.features.GeoJson)]
    link_layer.layer_name = "Links"
    if default:
        m = default(m)
    folium.plugins.Search(layer=link_layer, geom_type="LineString", placeholder="CML ID", collapsed=True, search_label="cml_id").add_to(m)
    if cp_cml_id_on_rclick:
        rclick_cp_cml_id(m)
    return m

In [ ]:
#| exec_doc
m = explore_links(cml)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

## Visualize link-gauge groups

Links and gauges are grouped by colours. As before right clicking on any link will return the `cml_id`.

In [ ]:
#| exec_doc
#| export
def explore_link_gauge_groups(
    cml: xr.Dataset | xr.DataArray, # CML data containing the id of the assigned (e.g. nearest) gauge
    gauges: xr.Dataset | xr.DataArray, # Gauges data with the same id as in CML
    default: Callable = setup_default_map, # Function to set up a default map including for example basemaps or controls
    id_to_color: Callable | dict = None, # Function or dict that allows to map gauge id to a color
    cp_cml_id_on_rclick: bool = True, # Enable copying cml_id to clipboard when right clicking on a link
    **explore_kwargs
) -> folium.Map:
    dims = [d for d in gauges.dims if d != "time"]
    if len(dims) != 1:
        raise ValueError(f"Expected exactly one non-time dimension, found {len(dims)}: {dims}")
    id_dim = dims[0]
    
    cml_df = cml.coords.to_dataset().drop_vars("time").to_dataframe().reset_index()
    geom = [LineString([(l.site_0_lon, l.site_0_lat), (l.site_1_lon, l.site_1_lat)]) for _, l in cml_df.iterrows()]
    cml_gdf = gpd.GeoDataFrame(cml_df[["cml_id", id_dim]], geometry=geom, crs="EPSG:4326")
    cml_gdf = cml_gdf.drop_duplicates(subset="cml_id", keep="first")

    gauges_df = gauges.id.to_dataframe().reset_index(drop=True)
    gauges_gdf = gpd.GeoDataFrame(gauges_df[[id_dim]], geometry=[Point(g.lon, g.lat) for _, g in gauges_df.iterrows()], crs="EPSG:4326")

    ens = pd.concat([cml_gdf, gauges_gdf]).sort_values(id_dim)
    explore_kwargs = {"column": id_dim, "categorical": True, "tiles": None, "marker_kwds": {"radius": 8}, **explore_kwargs}
    if id_to_color is not None:
        sorted_colors = [id_to_color(g) if callable(id_to_color) else id_to_color[g] for g in sorted(ens[id_dim].unique())]
        explore_kwargs["cmap"] = ListedColormap(sorted_colors)
    m = ens.explore(**explore_kwargs)

    gauges_layer = [c for c in m._children.values() if isinstance(c, folium.features.GeoJson)][-1]
    gauges_layer.layer_name = "Link-Gauge groups"
    if default:
        m = default(m)
    if cp_cml_id_on_rclick:
        rclick_cp_cml_id(m)
    return m

We will start by opening a sample that contains the nearest gauge to each link based on the distance to its center. You can check how this sample was created on the merging doc.

In [ ]:
#| exec_doc
cml_with_gauges = open_cml_sample("with_nearest_gauge")

In [ ]:
#| exec_doc
m = explore_link_gauge_groups(cml_with_gauges, gauges)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

Geopandas explore will assign colors automatically which is very useful. However in some cases we might want to add custom colors.

In [ ]:
#| exec_doc
color_mapper = {
'Campus_2_Universite': '#1f77b4',
'Ecole_Saint_Andre_PK11': '#ff7f0e',
'Eglise_Pie_X_PK14': '#2ca02c',
'Hopital_des_Soeurs_Logpom': '#d62728',
'Lycee_NYALLA': '#9467bd',
'Meteo_IUT': '#8c564b',
'Piscine_CITE_SIC': '#e377c2',
'SOCATUR': '#ffd000ff'
}
m = explore_link_gauge_groups(cml_with_gauges, gauges, id_to_color=color_mapper)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

## Visualize CML sublinks

Sublinks are colored by frequency using the electromagnetic spectrum coloring (red for lowest frequencies to violet for highest). Sublinks within the same CML are slightly offset for visibility. Optionally shows transmission direction arrows It also supports right-click to copy IDs that can then be used to select the sublink of interest from the dataset (see the example at link visualization section).

In [ ]:
#| exporti
def add_sense(m: folium.Map, layer_name: str = "Sublinks"):
    """ Add directional arrows to sublinks showing the transmission sense (direction from transmitter to receiver) """
    layer = [l for l in m._children.values() if isinstance(l, folium.features.GeoJson) and l.layer_name == layer_name]
    if len(layer) != 1:
        raise ValueError(f"{len(layer)} matching layers found.  There should be only one matching layer")
    layer = layer[0]

    fg = folium.FeatureGroup(name=layer_name + "_sense", show=False).add_to(m)
    for feat in layer.data["features"]:
        coords = feat["geometry"]["coordinates"]
        coords = [[coords[0][1], coords[0][0]], [coords[1][1], coords[1][0]]]
        hidden_pl = folium.PolyLine(coords, weight=0).add_to(fg) # Hidden lines required to add sens over them

        ws = "    "
        arrow = 2*ws + ">" + 2*ws if not feat["properties"]["transmitter"] else 3*ws + "<" + ws
        sense = folium.plugins.PolyLineTextPath(hidden_pl, arrow, repeat=True, offset=8, attributes={"font-weight": "bold", "font-size": "24", "fill": "red"})
        sense.add_to(fg)
    return m

In [ ]:
#| exporti
def rclick_cp_ids(m: folium.Map):
    """ Copy cml_id and sublink_id as a python dict to clipboard when right clicking on a sublink """
    js_code = """
    <script>
    document.addEventListener('DOMContentLoaded', function() {
        setTimeout(function() {
            var map = Object.values(window).find(v => v instanceof L.Map);
            map.eachLayer(function(layer) {
                if (layer.feature && layer.feature.properties && layer.feature.properties.cml_id) {
                    layer.on('contextmenu', function(e) {
                        L.DomEvent.stopPropagation(e);
                        L.DomEvent.preventDefault(e);
                        var cmlId = e.target.feature.properties.cml_id;
                        var sublinkId = e.target.feature.properties.sublink_id;
                        var jsonObj = JSON.stringify({cml_id: cmlId, sublink_id: sublinkId});
                        navigator.clipboard.writeText(jsonObj);
                        console.log('Copied:', jsonObj);
                    });
                }
            });
        }, 1000);
    });
    </script>
    """

    m.get_root().html.add_child(folium.Element(js_code))

In [ ]:
convert_sublinks_to_gdf?

In [ ]:
#| exec_doc
#| export
def explore_sublinks(
    cml: xr.Dataset | xr.DataArray, # CML in OpenSense standard
    default: Callable = setup_default_map, # Function to set up a default map including for example basemaps or controls
    add_transmission_sense: bool = True, # If true it will overlay an arrow over each sublink showing the sense of the signal
    cp_id_on_rclick: bool = True, # Enable copying cml_id and sublink_id when right clicking on a sublink
    offset: float = 2e-4, # Perpendicular offset in degrees (2e-4 ~ 20m in the equator) in order to avoid overlap in visualization
    **explore_kwargs
    ) -> folium.Map:
    """ Geopandas [explore](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.explore.html) wrapper to visualize CML sublink structure by coloring on frequency """
    sl_gdf = convert_sublinks_to_gdf(cml, offset=offset)
    sl_gdf = sl_gdf.filter(regex="^(?!.*(lat|lon))")
    if "transmitter" in sl_gdf:
        sl_gdf["transmitter"] = sl_gdf["transmitter"].astype(int)
    explore_kwargs = {"tiles": None, "cmap": "gist_rainbow", "vmin": 8000, "vmax": 18000, "tiles": None, **explore_kwargs}
    m = sl_gdf.explore("frequency", **explore_kwargs)
    sublink_layer, = [layer for name, layer in m._children.items() if name.startswith("geo_json_") and isinstance(layer, folium.features.GeoJson)]
    sublink_layer.layer_name = "Sublinks"

    if add_transmission_sense:
        assert "transmitter" in sl_gdf, ValueError("transmitter coordinate is required to add sense")
        add_sense(m, "Sublinks")
    if default:
        m = default(m)
    if cp_id_on_rclick:
        rclick_cp_ids(m)

    return m

In [ ]:
#| exec_doc
m = explore_sublinks(cml)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

## Visualize sublinkwise data

In [ ]:
#| exec_doc
#| export
def explore_data(
    cml: xr.DataArray, # CML raw or derived data in OpenSense standard
    vmin: float = None, # Minimum value for the colormap
    vmax: float = None, # Maximum value for the colormap
    cmap: ListedColormap = plt.cm.viridis, # Matplotlib colormap
    m: folium.Map = None, # Map where animation will be added. If not given, setup_default_map will be used
    offset: float = 2e-4, # Perpendicular offset in degrees (2e-4 ~ 20m in the equator) in order to avoid overlap in visualization
) -> folium.Map:
    def assign_style(v, normalizer, cmap):
        return {"color": to_hex(cmap(normalizer(v))), "weight": 2} 
    if not cml.name:
        raise NameError("You must provide a dataset with a valid name")
    var = cml.name
    gdf = convert_sublinks_to_gdf(cml, only_meta=False, offset=offset)
    gdf = gdf.filter(regex="^(?!.*(lat|lon))")
    gdf = gdf.dropna(subset=var)

    vmin = vmin if vmin is not None else gdf[var].min()
    vmax = vmax if vmax is not None else gdf[var].max()
    normalizer = Normalize(vmin=vmin, vmax=vmax)
    gdf["style"] = gdf[var].apply(assign_style, args=(normalizer, cmap))

    gdf["time"] = gdf['time'].dt.strftime('%Y-%m-%dT%H:%M:%S')
    gdf["times"] = gdf["time"].apply(lambda t: [t, t])
    gdf = gdf.sort_values(by=["time", "cml_id", "sublink_id"])
    gdf.drop(columns="time", inplace=True)

    m = m or setup_default_map(folium.Map(tiles=None, zoom_start=get_zoom_start(cml), location=get_center(cml)))
    explore_sublinks(cml.to_dataset(), show=False) # Remove .to_dataset() when explore_sublinks will accept xr.DataArray
    folium.plugins.TimestampedGeoJson(
        gdf.__geo_interface__,
        period="PT15M",
        add_last_point=False,
        auto_play=False,
        loop=False,
        max_speed=1,
        loop_button=True,
        time_slider_drag_update=True,
    ).add_to(m)
    return m

In [ ]:
#| exec_doc
cml = open_cml_sample()

:::{.callout-tip}
If you are showing variables such as attenuation that will depend on several factors such as baseline, length, ... remember to normalize the values. This step is not necessary when showing the rainfall.
:::
For the sake of this example, we will normalise the variable based on data from two consecutive days. To avoid potential issues due to outliers, we will use quantiles 0.1 and 0.9.

In [ ]:
#| exec_doc
cml = cml.sel(time=slice("2019-07-17T00:00", "2019-07-19T00:00"))
att = cml["tsl_max"] - cml["rsl_min"]
att.name = "att"
q = att.quantile([0.1, 0.9], dim="time")
att = (att - q.sel(quantile=0.1, drop=True)) / (q.sel(quantile=0.9, drop=True) - q.sel(quantile=0.1, drop=True))

:::{.callout-warning}
Given the current state of development, we recommend plotting a maximum of one day of data at 15-minute intervals (approximately 96 timesteps). You can use the [Romulo plot](https://rainsmore.github.io/raincell/plot.romulo.html) to identify the periods of interest. Please remember to toggle the visibility of the metadata before playing to achieve a more fluid animation.
:::
In this case, to avoid performance issues, we will only show few hours of data.

In [ ]:
#| exec_doc
m = explore_data(att.sel(time=slice("2019-07-17T18:00", "2019-07-18T03:00")), vmin=0, vmax=1)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

## How to join visualizations?

### Stacking layers together. Example of the sublink and gauge data

In [ ]:
#| exec_doc
m = explore_sublinks(cml, default=None)
m = explore_gauges(gauges, default=None, m=m)
m = setup_default_map(m)

In [ ]:
#| exec_doc
#| code-fold: true
m.options['scrollWheelZoom'] = False
m

### Visualizing data side by side. Example of sublink metadata and link-gauge groups

We will import some helper functions

In [ ]:
#| exec_doc
m = folium.plugins.DualMap(zoom_start=get_zoom_start(cml), location=get_center(cml), tiles=None)
lm, rm = m.m1, m.m2
lm = explore_sublinks(cml, default=None, m=lm)
rm = explore_link_gauge_groups(cml_with_gauges, gauges, m=rm, default=None)
m = setup_default_map(m)

In [ ]:
#| exec_doc
#| code-fold: true
lm.options['scrollWheelZoom'] = False
rm.options['scrollWheelZoom'] = False
m

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()